# 📊 00 · 业务指标与描述性统计

> 数据分析岗面试第一关：**指标口径 + 描述统计**。本节把「均值/中位数/方差/分位数」和「DAU/留存率/转化率/ARPU/GMV 拆解」讲清楚，并用 numpy/pandas 手写对照。

本节风格：公式 → 手写 → numpy/pandas 对照 → 业务指标演练 → 自测。

## 📋 本节要解决的问题

| # | 问题 | 工具 |
|---|---|---|
| 1 | 描述一批数据用什么指标？各有什么陷阱？ | 均值/中位数/众数/方差/分位数 |
| 2 | 「平均数被拉高」是怎么发生的？ | 偏态与离群值 |
| 3 | 业务上常看的指标口径是什么？ | DAU/MAU、留存率、转化率、ARPU、GMV |
| 4 | 指标怎么拆解定位问题？ | 杜邦式拆解 GMV=流量×转化×客单价 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
np.random.seed(42)


## 1️⃣ 描述性统计：手写 vs numpy

一组销售金额（元）：

In [2]:
sales = np.array([120, 88, 150, 99, 210, 165, 999, 132, 145, 108])
print('原始数据:', sales)

原始数据: [120  88 150  99 210 165 999 132 145 108]


In [3]:
# 手写
n = len(sales)
mean_manual = sum(sales) / n
sorted_s = sorted(sales)
median_manual = (sorted_s[n//2-1] + sorted_s[n//2]) / 2 if n % 2 == 0 else sorted_s[n//2]
var_manual = sum((x - mean_manual)**2 for x in sales) / n
std_manual = var_manual ** 0.5

# numpy 对照
mean_np, med_np, var_np, std_np = np.mean(sales), np.median(sales), np.var(sales), np.std(sales)

print(f'手写: 均值={mean_manual:.2f} 中位数={median_manual:.2f} 方差={var_manual:.2f} 标准差={std_manual:.2f}')
print(f'numpy: 均值={mean_np:.2f} 中位数={med_np:.2f} 方差={var_np:.2f} 标准差={std_np:.2f}')
assert abs(mean_manual - mean_np) < 1e-9 and abs(std_manual - std_np) < 1e-9
print('✅ 手写与 numpy 完全一致')

手写: 均值=221.60 中位数=138.50 方差=68281.84 标准差=261.31
numpy: 均值=221.60 中位数=138.50 方差=68281.84 标准差=261.31
✅ 手写与 numpy 完全一致


## 2️⃣ 均值 vs 中位数：离群值陷阱

`999` 这个离群值把均值拉高了，中位数更稳健：

In [4]:
print(f'均值={np.mean(sales):.1f}  中位数={np.median(sales):.1f}')
print('→ 均值被离群值拉高', np.mean(sales) - np.median(sales), '元')
assert np.mean(sales) > np.median(sales)  # 右偏：均值>中位数

# 用箱线图看分布
plt.figure(figsize=(7, 3))
plt.boxplot(sales, vert=False)
plt.title('销售金额分布（箱线图可见离群值 999）')
plt.xlabel('金额（元）')
plt.show()

均值=221.6  中位数=138.5
→ 均值被离群值拉高 83.1 元


C:\Users\24260\AppData\Local\Temp\ipykernel_32376\2102711236.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3️⃣ 分位数与四分位距 IQR

分位数用于描述分布位置，IQR 是常用离群值判据：

In [5]:
q25, q50, q75 = np.percentile(sales, [25, 50, 75])
iqr = q75 - q25
lower, upper = q25 - 1.5*iqr, q75 + 1.5*iqr
print(f'Q1={q25:.1f} Q2={q50:.1f} Q3={q75:.1f} IQR={iqr:.1f}')
print(f'IQR 离群区间: 低于 {lower:.1f} 或高于 {upper:.1f}')
print('疑似离群:', sales[(sales < lower) | (sales > upper)])
assert (sales[(sales < lower) | (sales > upper)] == np.array([999])).all()
print('✅ IQR 正确识别出 999')

Q1=111.0 Q2=138.5 Q3=161.2 IQR=50.2
IQR 离群区间: 低于 35.6 或高于 236.6
疑似离群: [999]
✅ IQR 正确识别出 999


## 4️⃣ 业务指标口径

模拟一个 App 一周的 DAU 与新增用户：

In [6]:
days = pd.date_range('2025-01-06', periods=7)
dau = pd.Series([12000, 11800, 12400, 11900, 12300, 15600, 15200], index=days)
new_users = pd.Series([900, 950, 880, 920, 910, 1200, 1180], index=days)

print('DAU 均值:', dau.mean().round(0))
print('新增均值:', new_users.mean().round(0))

# 次日留存率（假设第二天回来的人数）
retained = pd.Series([540, 570, 520, 550, 545, 720, 700], index=days)
retention = (retained / new_users).mean()
print('次日留存率:', f'{retention:.1%}')

# ARPU = 收入 / 活跃用户
daily_revenue = pd.Series([48000, 47000, 50000, 47500, 49000, 62000, 61000], index=days)
arpu = daily_revenue.mean() / dau.mean()
print('ARPU（每活跃用户收入）:', f'{arpu:.2f} 元')

DAU 均值: 13029.0
新增均值: 991.0
次日留存率: 59.7%
ARPU（每活跃用户收入）: 4.00 元


## 5️⃣ GMV 拆解：流量 × 转化率 × 客单价

把总指标拆成可定位的因子乘积，是面试高频题：

In [7]:
pv = 100_000      # 页面浏览量
conversion = 0.02  # 转化率
avg_price = 150    # 客单价
gmv = pv * conversion * avg_price
print(f'GMV = {pv} × {conversion} × {avg_price} = {gmv:,.0f} 元')
assert gmv == 300_000

# 若转化率提升到 0.03
conversion2 = 0.03
gmv2 = pv * conversion2 * avg_price
print(f'转化率提升 50% → GMV = {gmv2:,.0f} 元，增幅 {gmv2/gmv-1:.0%}')

# 可视化拆解
factors = ['PV', '转化率', '客单价']
vals = [pv, conversion*100, avg_price]
plt.figure(figsize=(6, 4))
plt.bar(factors, vals, color=['#60a5fa', '#f59e0b', '#34d399'])
plt.title('GMV 三因子（量纲不同，仅示意相对大小）')
plt.show()

GMV = 100000 × 0.02 × 150 = 300,000 元
转化率提升 50% → GMV = 450,000 元，增幅 50%


C:\Users\24260\AppData\Local\Temp\ipykernel_32376\1972480099.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6️⃣ 直方图：分布长什么样

描述统计要配合图形：

In [8]:
rng = np.random.default_rng(7)
login_minutes = np.concatenate([rng.normal(20, 5, 400), rng.normal(55, 10, 100)])

plt.figure(figsize=(7, 4))
plt.hist(login_minutes, bins=30, color='#60a5fa', edgecolor='white')
plt.axvline(np.mean(login_minutes), color='red', ls='--', label=f'均值 {np.mean(login_minutes):.1f}')
plt.axvline(np.median(login_minutes), color='green', ls='--', label=f'中位数 {np.median(login_minutes):.1f}')
plt.xlabel('单次使用时长（分钟）')
plt.ylabel('用户数')
plt.title('使用时长分布：双峰（均值>中位数，右偏）')
plt.legend()
plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_32376\2676356257.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 📝 自测清单

- [ ] 能解释「均值 vs 中位数 vs 众数」各自适用场景与离群值敏感性
- [ ] 能手写方差/标准差/IQR，并说出 1.5×IQR 规则的由来
- [ ] 能说出 DAU/MAU、次日留存率、转化率、ARPU、GMV 的口径
- [ ] 能把 GMV 拆成 流量×转化率×客单价，并说明每个因子的提升手段